In [ ]:
import mdtraj as md
import numpy as np
import pandas as pd
from pathlib import Path
import glob
import os
from collections import defaultdict
import warnings

# Suppress warnings that might clutter the notebook output
warnings.filterwarnings('ignore', category=UserWarning)

# ==============================================================================
# USER CONFIGURABLE PARAMETERS
# ==============================================================================

# --- File Paths and Patterns ---
# Root directory containing the 'repl_*' subfolders with topology and trajectory files.
SEARCH_ROOT_DIR = "."

# Topology and trajectory file naming patterns inside each 'repl' directory.
TOPOLOGY_PATTERN = "state0.{repl}.pdb"
TRAJECTORY_PATTERN = "center.{repl}.xtc"

# The Excel output file name pattern. This will be an absolute path if the pattern is absolute.
OUTPUT_XLSX_PATTERN = "deltaSASA_summary_repl_{repl}.xlsx"

# ADDED: New base directory for all output files, as requested.
OUTPUT_ROOT_DIR = "."

# --- Chains and Residues to Evaluate ---
# If None, the code will auto-detect chain 0 and chain 1 residues.
CHAIN_A_RESIDS = None # Residue indices to evaluate in Chain A
CHAIN_B_RESIDS = None # Residue indices to evaluate in Chain B

# --- Analysis Parameters ---
# The probe radius used for Shrake-Rupley SASA calculation (in nm).
PROBE_RADIUS_NM = 0.14
# The number of points used to approximate the sphere for SASA calculation.
N_SPHERE_POINTS = 960
# Frames to load per chunk. Set to None to load the whole trajectory.
CHUNK_SIZE = 200

# --- Output Sheet Names and Filtering Thresholds ---
SHEET_RAW_DATA = 'delta_sasa_raw'

# ==============================================================================
# CORE LOGIC
# ==============================================================================

def create_pdb_to_mdtraj_map_robust(topology):
    """
    Creates a dictionary mapping (mdtraj_chain_index, PDB_residue_number)
    to the mdtraj zero-based residue index.
    """
    pdb_to_mdtraj_robust = {}
    for residue in topology.residues:
        pdb_to_mdtraj_robust[(residue.chain.index, residue.resSeq)] = residue.index
    return pdb_to_mdtraj_robust

def find_replica_directories(search_root: str):
    """
    Finds and returns a sorted list of all 'repl_*' subdirectories within the search root.
    """
    search_path = Path(search_root)
    replica_dirs = sorted(
        str(d) for d in search_path.glob("repl_*") if d.is_dir()
    )
    if not replica_dirs:
        raise FileNotFoundError(f"No 'repl_*' directories found inside {search_root}")
    return replica_dirs

def calculate_residue_stats(unbound_arr, bound_arr, resid_index, chain_label, topology_traj):
    """
    Computes statistical metrics (mean, median, SD, and occupancy fractions)
    for the change in Solvent Accessible Surface Area (ΔSASA).
    
    NOTE: unbound_arr and bound_arr are now guaranteed to be single, 
    concatenated NumPy arrays.
    """
    unbound = unbound_arr
    bound = bound_arr
    delta_sasa = unbound - bound

    mean_unbound = np.mean(unbound)
    mean_bound = np.mean(bound)
    mean_delta = np.mean(delta_sasa)
    median_unbound = np.median(unbound)
    median_bound = np.median(bound)
    median_delta = np.median(delta_sasa)
    std_dev_delta = np.std(delta_sasa)

    total_frames = len(delta_sasa)

    # Occupancy calculations remain, as they are part of the 'raw' data output.
    occ_0_1_0_3 = np.sum((delta_sasa >= 0.1) & (delta_sasa < 0.3)) / total_frames if total_frames > 0 else np.nan
    occ_0_3_0_5 = np.sum((delta_sasa >= 0.3) & (delta_sasa <= 0.5)) / total_frames if total_frames > 0 else np.nan
    occ_gt_0_5 = np.sum(delta_sasa > 0.5) / total_frames if total_frames > 0 else np.nan

    residue_obj = topology_traj.topology.residue(resid_index)

    return {
        'chain': chain_label,
        'residue_mdtraj_idx': resid_index,
        'residue_pdb_resseq': residue_obj.resSeq,
        'resname': residue_obj.name,
        'mean_sasa_unbound_nm2': mean_unbound, 'mean_sasa_bound_nm2': mean_bound, 'mean_delta_sasa_nm2': mean_delta,
        'median_sasa_unbound_nm2': median_unbound, 'median_sasa_bound_nm2': median_bound, 'median_delta_sasa_nm2': median_delta,
        'sd_delta_sasa_nm2': std_dev_delta,
        'occupancy_0.1_0.3': occ_0_1_0_3, 'occupancy_0.3_0.5': occ_0_3_0_5, 'occupancy_gt_0.5': occ_gt_0_5
    }


def run_sasa_analysis():
    """
    Main execution function. Traverses replica directories, loads trajectories,
    calculates per-residue SASA in unbound and bound states, determines ΔSASA,
    and saves the results into a single Excel file per replica.
    """
    try:
        replica_directories = find_replica_directories(SEARCH_ROOT_DIR)
    except FileNotFoundError as e:
        # UNIFIED GLOBAL ERROR
        print(f"Global ERROR ❌ - {e}")
        return

    # UNIFIED GLOBAL START STATUS
    print(f"Global INFO - Found {len(replica_directories)} replica directories. Starting ΔSASA calculation analysis.")

    # Estimate total chunks for printing (unchanged)
    estimated_total_chunks = None
    if CHUNK_SIZE is not None and replica_directories:
        try:
            first_replica_dir = Path(replica_directories[0])
            replica_num = first_replica_dir.name.split("_")[-1]
            trajectory_file = first_replica_dir / TRAJECTORY_PATTERN.format(repl=replica_num)

            if trajectory_file.exists():
                # Load first frame to get total frame count approximation
                t = md.load(str(trajectory_file), frame=0)
                total_frames_approx = t.n_frames
                estimated_total_chunks = (total_frames_approx + CHUNK_SIZE - 1) // CHUNK_SIZE
                print(f"Global INFO - Estimated total frames: {total_frames_approx}, chunks per replica: {estimated_total_chunks}")
                del t # Free memory
        except Exception:
            estimated_total_chunks = None

    # Prepare the new output directory
    output_root_path = Path(OUTPUT_ROOT_DIR)
    try:
        # Ensure the new output directory exists
        os.makedirs(output_root_path, exist_ok=True)
    except Exception as e:
        print(f"Global ERROR ❌ - Could not create output directory {OUTPUT_ROOT_DIR}: {e}. Exiting.")
        return

    for replica_dir_str in replica_directories:
        replica_dir = Path(replica_dir_str)
        replica_num = replica_dir.name.split("_")[-1]

        # UNIFIED MAJOR START
        print(f"\n>>> Starting ΔSASA Calculation for Replica [{replica_num}] (Directory: {replica_dir})")

        topology_file = replica_dir / TOPOLOGY_PATTERN.format(repl=replica_num)
        trajectory_file = replica_dir / TRAJECTORY_PATTERN.format(repl=replica_num)

        output_xlsx_base = OUTPUT_XLSX_PATTERN.format(repl=replica_num)
        
        # ADJUSTMENT: Construct the output path using the new OUTPUT_ROOT_DIR
        # If OUTPUT_XLSX_PATTERN is already an absolute path (unlikely given its definition), this is redundant.
        # Given the request, we enforce the new location.
        output_xlsx_path = str(output_root_path / output_xlsx_base)

        try:
            topology_traj = md.load(str(topology_file))
        except Exception as e:
            # UNIFIED REPLICA ERROR
            print(f"Replica [{replica_num}] ERROR ❌ - Could not load topology file: {e}. Skipping replica.")
            continue
            
        chains = list(topology_traj.topology.chains)
        chain_a_resids_local = [r.index for r in chains[0].residues] if CHAIN_A_RESIDS is None else list(CHAIN_A_RESIDS)
        chain_b_resids_local = [r.index for r in chains[1].residues] if CHAIN_B_RESIDS is None else list(CHAIN_B_RESIDS)

        chain_a_atoms = [a.index for r in topology_traj.topology.residues if r.index in chain_a_resids_local for a in r.atoms]
        chain_b_atoms = [a.index for r in topology_traj.topology.residues if r.index in chain_b_resids_local for a in r.atoms]
        protein_atoms = chain_a_atoms + chain_b_atoms

        combined_resids = list(chain_a_resids_local) + list(chain_b_resids_local)

        atompos_in_protein = {atom_idx: pos for pos, atom_idx in enumerate(protein_atoms)}
        atompos_in_chainA = {atom_idx: pos for pos, atom_idx in enumerate(chain_a_atoms)}
        atompos_in_chainB = {atom_idx: pos for pos, atom_idx in enumerate(chain_b_atoms)}

        protein_res_atompos = []
        for res_idx in combined_resids:
            res = topology_traj.topology.residue(res_idx)
            protein_res_atompos.append([atompos_in_protein[a.index] for a in res.atoms])

        chain_a_res_atompos = []; chain_b_res_atompos = []
        for res_idx in chain_a_resids_local:
            res = topology_traj.topology.residue(res_idx)
            chain_a_res_atompos.append([atompos_in_chainA[a.index] for a in res.atoms])

        for res_idx in chain_b_resids_local:
            res = topology_traj.topology.residue(res_idx)
            chain_b_res_atompos.append([atompos_in_chainB[a.index] for a in res.atoms])

        # 🟢 OPTIMIZATION: These lists now store NumPy arrays (chunks)
        # instead of many individual floats, reducing Python list overhead.
        complex_res_values_chunks = [ [] for _ in combined_resids ]
        chain_a_res_values_chunks = [ [] for _ in chain_a_resids_local ]
        chain_b_res_values_chunks = [ [] for _ in chain_b_resids_local ]

        total_frames = 0

        if CHUNK_SIZE is None:
            iterator = [md.load(str(trajectory_file), top=str(topology_file))]
            chunk_count_str = ""
        else:
            iterator = md.iterload(str(trajectory_file), top=str(topology_file), chunk=CHUNK_SIZE)
            chunk_count_str = f"/{estimated_total_chunks}" if estimated_total_chunks else ""


        for chunk_index, chunk in enumerate(iterator):
            n_frames_chunk = chunk.n_frames
            total_frames += n_frames_chunk

            # UNIFIED CHUNK PRINTING
            print(f"Replica [{replica_num}] - Chunk [{chunk_index+1}{chunk_count_str}]: Calculating SASA for {n_frames_chunk} frames...", end=' ')

            # Calculate SASA for complex (Bound state)
            chunk_complex = chunk.atom_slice(protein_atoms)
            sasa_complex_chunk = md.shrake_rupley(chunk_complex, probe_radius=PROBE_RADIUS_NM, n_sphere_points=N_SPHERE_POINTS)

            # Calculate SASA for Chain A in isolation (Unbound state)
            chunk_chainA = chunk.atom_slice(chain_a_atoms)
            sasa_chainA_chunk = md.shrake_rupley(chunk_chainA, probe_radius=PROBE_RADIUS_NM, n_sphere_points=N_SPHERE_POINTS)

            # Calculate SASA for Chain B in isolation (Unbound state)
            chunk_chainB = chunk.atom_slice(chain_b_atoms)
            sasa_chainB_chunk = md.shrake_rupley(chunk_chainB, probe_radius=PROBE_RADIUS_NM, n_sphere_points=N_SPHERE_POINTS)

            # Accumulate results for complex (Bound)
            for i, atompos_list in enumerate(protein_res_atompos):
                res_sasa_chunk = sasa_complex_chunk[:, atompos_list].sum(axis=1)
                # 🟢 OPTIMIZED: Append the NumPy array directly
                complex_res_values_chunks[i].append(res_sasa_chunk)

            # Accumulate results for Chain A (Unbound)
            for i, atompos_list in enumerate(chain_a_res_atompos):
                res_sasa_chunk = sasa_chainA_chunk[:, atompos_list].sum(axis=1)
                # 🟢 OPTIMIZED: Append the NumPy array directly
                chain_a_res_values_chunks[i].append(res_sasa_chunk)

            # Accumulate results for Chain B (Unbound)
            for i, atompos_list in enumerate(chain_b_res_atompos):
                res_sasa_chunk = sasa_chainB_chunk[:, atompos_list].sum(axis=1)
                # 🟢 OPTIMIZED: Append the NumPy array directly
                chain_b_res_values_chunks[i].append(res_sasa_chunk)

            # Explicitly delete large chunk variables immediately after use to free memory
            del chunk, chunk_complex, chunk_chainA, chunk_chainB, sasa_complex_chunk, sasa_chainA_chunk, sasa_chainB_chunk
            print("done.")

        print(f"Replica [{replica_num}] INFO - Total frames processed: {total_frames}")

        results_rows = []

        # Process Chain A residues
        for i, resid_i in enumerate(chain_a_resids_local):
            # 🟢 OPTIMIZED: Concatenate all chunks into a single NumPy array just before stats calculation
            unbound = np.concatenate(chain_a_res_values_chunks[i])
            # The complex-bound data for Chain A residues start at index 0 of complex_res_values
            bound = np.concatenate(complex_res_values_chunks[i])
            stats = calculate_residue_stats(unbound, bound, resid_i, 'A', topology_traj)
            results_rows.append(stats)

        # Process Chain B residues
        offset = len(chain_a_resids_local)
        for i, resid_i in enumerate(chain_b_resids_local):
            # 🟢 OPTIMIZED: Concatenate all chunks into a single NumPy array just before stats calculation
            unbound = np.concatenate(chain_b_res_values_chunks[i])
            # The complex-bound data for Chain B residues start after Chain A data
            bound = np.concatenate(complex_res_values_chunks[offset + i])
            stats = calculate_residue_stats(unbound, bound, resid_i, 'B', topology_traj)
            results_rows.append(stats)

        # Create DataFrames and Apply Final Formatting
        df_raw_data = pd.DataFrame(results_rows)

        column_order = [
            'chain', 'residue_mdtraj_idx', 'residue_pdb_resseq', 'resname',
            'mean_sasa_unbound_nm2', 'mean_sasa_bound_nm2', 'mean_delta_sasa_nm2',
            'median_sasa_unbound_nm2', 'median_sasa_bound_nm2', 'median_delta_sasa_nm2',
            'sd_delta_sasa_nm2',
            'occupancy_0.1_0.3', 'occupancy_0.3_0.5', 'occupancy_gt_0.5'
        ]

        column_order = [c for c in column_order if c in df_raw_data.columns]
        df_raw_data = df_raw_data[column_order]

        # --- Save results to Excel ---
        try:
            with pd.ExcelWriter(output_xlsx_path, engine='xlsxwriter') as writer:
                df_raw_data.to_excel(writer, sheet_name=SHEET_RAW_DATA, index=False)

            # UNIFIED REPLICA SUCCESS
            print(f"Replica [{replica_num}] SUCCESS ✅ - Results saved to {Path(output_xlsx_path).name} in {output_root_path} ({SHEET_RAW_DATA} sheet ONLY)")

        except Exception as e:
            # Fallback to CSV if Excel writing fails
            csv_out_raw = output_xlsx_path.replace('.xlsx', '_raw.csv')
            df_raw_data.to_csv(csv_out_raw, index=False)

            # UNIFIED REPLICA FALLBACK ERROR
            print(f"Replica [{replica_num}] ERROR ❌ - Could not write XLSX file: {e}. Saved CSV fallback to {Path(csv_out_raw).name}")


# Execute the main function when the script runs
if __name__ == "__main__":
    run_sasa_analysis()